# CFPB Seed Pool Build v05.2

Fail-closed builder for the 2,000-row proportional core plus coverage supplement. It cannot run until decision v04 has been finalized after C-lite fuzzy review. Output remains a privacy-QA-pending candidate.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, json, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
RUN_ID = "run_20260713T145423Z"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/FinDisputeEval")
else:
    here = Path.cwd().resolve()
    ROOT = next((p for p in (here, *here.parents) if (p / "WORK_PROGRESS.md").exists()), None)
    if ROOT is None: raise FileNotFoundError("Open the repository")
EDA = ROOT / "outputs/data_pipeline/cfpb_seed_source_eda/eda_v051" / RUN_ID
AUDIT = ROOT / "dataset/curated/annotations/cfpb_seed_v05_audit" / RUN_ID
DECISION = AUDIT / "seed_v052_decision_record_v04.json"
OUTPUT = ROOT / "dataset/curated/seed_pools/cfpb_dispute/seed_v052"
if not DECISION.exists(): raise FileNotFoundError("Complete fuzzy v04 review first: " + str(DECISION))
record = json.loads(DECISION.read_text(encoding="utf-8"))
if record.get("record_version") != "v04" or not record.get("build_gate", {}).get("passed"):
    raise ValueError("Decision v04 build gate is closed")
print({"eda": str(EDA), "decision": str(DECISION), "output": str(OUTPUT)})


In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas>=2.2,<3", "numpy>=1.26,<3", "pyarrow>=16"])


In [ ]:
embedded = json.loads("{\"cfpb_seed_v05.py\": [\"1f0fcfac9dda5eb8b122fc3007b43ad3f185c3667b9c309cca89cfeef7c0d23a\", \"H4sIAAAAAAAC/61cfXPbxtH/n58CQScdoKZQyo3TVmPWdW15qk5qu37JPE8pDAYkjyRiEIDxIpmRlc/e3b134PiiTDwZhTzc7e3u7e3+du9A3/dfskXWZGVxtk5btvRevHr7D+89g083kydek26rnNXRaPRhw+Q3LytaVrQwJs3znbdJG68ovS1rN+WyzMt1tkhzb8lWaZe3TeRdtd6iLJpuyxqvLGBAOqrSphEzpN0ya6E3Z8Kr2aKsl97tpmyYly4WrEKm6vK2QSJtmhXeNl1ssoKd1SxdpvOcjf71/s1rr+5yxicr2A3wiE9hPmR2w+qsbbwWJLiZfOc1KFzNcpY2LBr5vj8arepy6yXJqmu7miWJl22rsm69tCjKNkVBm9FItIG0mzyby68/NWUhP9eME1qmbbrIUcRGUlJNqgdrsy0zHtP3sYd/fy4LQalKW5xMdnsLX/mDdldlxVq2Py92ir+i21agYliSSjZVabGEBvivWo5Go3eX//l49e7yZfLy8sXV+6s3r5Pn7y6fv/em3t3Ig38++5Iu2mTZVTmsZMsaf8zbWwbLDw3JKt1meaYfrLqff945BtRsnTUtq+V3WJdd0myApyQvi7XutoQJQcuyocoy+TFPi3WXrpn6DmJ3QDRbJCuW4nKp2cg64XHyuYNVw+Z7FPYHkO0y+XD5fx+SF29++Pjv14akRVrXsL43LOmK2zqtKraU1PQjMEjmaE2LskBLHz7K2Rfx4H70/s3Hdy/2Tf+Cb4saTBs1C9vKU1SGZOv0dtj4W/K9TdvF5pA49OC/V2+B3yUj8d5eXSVvn3/4cPnOlOvy38+vfvAvYENEKFmWs6D2r+ez52f/nZz9NUq+fXQWP/q7/AqfryP8Et89Ht9fz/0xDrwKxXxv//nm9aVNjB7gv9oPnj395noZBs8urh89O5+dRddN/Cx8ht+DZ9fLuz/dX4fPZDN9F1/g83f3wTMc7BM9Od/796/7rItJaPgZ/H1Mf9VwOfL5ixdvPr7+sJfX6zmwBT6t7Ir2K/y//bpIayDb/AHaYd/OWf21KK+jZ19/h40wxfdcH5oG6GVscXv19sfvhppG8Zd352OQPgpRB+LLHHmFXTH6u3ZI9Jc8/o+TJ2CRq2x9QbRhVyZ1VyTLrL7grgdbpadOuKdO0EUZj8uurbq2N6YGB1RuE3S8Fxg8wFIeTx5/P/nz+Z9GgugK3G9VNm2SFVmbJEHD8lXonf3New3O8EIpAJsjgzGghLME/fYwAsdQ5jcsCO2xLvYtIq4Oe6lpaS0autkcORqhmM0mffzk+2SFS6V1N/YWm674lDTZz0yq6Nx7+hT0RFpo2porYZmtWYNPRRyKOD3B2G0G0hAfnO2yYkXg17Ds6P2BCEu3Wpm3G2CCT+xdTMXjCANnoLkJdX89f9RVGLJ4Nz51zcAZF/L5hn3hn5TcsPo5Iy+ffGK7QNnCWDu/JIMm4MKWuEp3eZkuQeSVf4fD7r/emUPu/YgV6JACv2tXZ3/xLX56ahLEQgeD2J5IxJEQnDAWiHhqIcKx2TJbtDNgb4xxNx579vf4QkxPOGZKACFC2k1gLAwqOWnZlzYg3iFqTSX3nP1sBZiqFWSiNWthGTlkSRCngY+8uw95OwdT0PIqzRtzveo0AxD1Y5p37LKuyzrwFa4TtDykJQAc4gQipBgwJ2/Krl6wBLcYRHHcIGBT30whpk+eROf+iZN+7jLYD17q0ag+6hMzy9YGtGeyoNpB1FnM+853SQq6xNADGHGm+lCrH18gcvRWgALx/4AdFY17Gr7NmgZxFBg/gBK2DPZhozPY720gZtNLJMYfkH6l8LU0iaxR0yKx5sK7E9/vpemi7V307ApF5EyjOCQ0yHOYbdMWQP6pVNcM/8TqGQgC0wTQxVZ00gD47VDdPpglGPEtqwO+6nKf+LZ3OCR+xo1MjgSxkQspM/67wXHe1GZGoH3akAluJskO9MoqwyGLHUNETudKJi/9WRzstfWuRxfXievS3ujEgx7IvqDIvAPmKjA5uCtiBd0yww9HOb4qgGq2HHJK9iCZ9Sg7IIqmF+SmN+YcS4dM9BDOEynMrmqA4YHT+nqhWOzkJbryRuE+d/IAmoRF/JJUrE74Q/TJ/r1GNY7EwhxEjTtrwCDhwP4Ag/MdyrHK6i3DyJ3VjTVM5SPUPUeDBk8vGsnOS0gS60RkI+bQfuqCFIw2mb+IjtjFRUVnOkrAsupySjGTqi4xXRN00nkJ2LvdgL/clPnSRQ3TpAtD91JG9iWps+ZTYiVVRgdy/Tz8gLrybjvow1UpSPWyM9VpCfxC/uGi1xh9TY5VNmcuAPcy7LD+XXkfElFJn2o1R/VTwoG2qjrbpriKDBkBuNMXkhV1tthsodyh00qrA6KmpgHhAbu6CMBzWN7VLpnvhkqxHfrYo+2UFfb+ijLYH01gOPNB2KJxPEYZTim0nOMgVu1xNNyVkKswYxVOMQxVfKXIBYBhYXkGVxEcXt5hhqjUg3Uff5+0d4e22ezEbRbv2WazfTspHuykmWPzxG7rndmWGrudjLvX/XBBUSegDVQG6dlaI0RielFn0CGmSAedhfrt/nvW9SPUGqqK7EWMA8zUo/tNfW+Ek+gO2uRKg4ooUCAoo1GktFgiIfl45vYvimNXPeYQeAS6kYuighOghAo85w1IhY+9m4zd+haGhkBfgI8pFpBwKS7dHg7Q/Lws8/AYR87R3raDzGzOiARLC82Fnvagz4ylUB9qE8IcQNQ0I9L0NE2PQ/Uz0oYkrVjR9QhzxQZWTyBPbmM1BNJ4nigcUyMvURwQwfaR78hzAOtgetxhnFGhmOh6yFgjXaJsU3MpSjLtEw/0hsfMEXCZdI2Gd4hnB2JwbBnRJJp4T6dO8tB8Hk0OrZcaFe2dzjCerliCKmaTsXce25asBBAeBvgfopj4MCe8WzQYxxnAWYAJwGPtzjm38oHG3CqAH5xajoz6w47NbOxfyYQDDAA/QywA+zmHfgf3s4NYNKCkliclgscYHODTeLYHnp7icfrUIiepPe7HCre9aOvA6i6ofgSp94H63igH6oKSTdAPOqH31Ds/EZbwkGSsBtQMM6xS05nUGvYEl5pjNR2r+kAwHvXCbuCCgg50NxSHU1SSTI5K0uOlJxJU588KtuZnAPvMTMzpQKeiHnXQnvoMDKho/RZeOf8Jynaak7TYBSg2z3FRYFIjz91BkQdYi6gTmMRB7i7VQK/HzYNUY+Ju6QYwvmB3Z5+HaMwYaNgi8kf+S/oIVfmUyJXHvVWdbqHKWy2jl1CFf4XfxpDXN5+o7T2cVrJmLDAaFUTxQCRtxJdRLxdPVytYIIZhDml4v+f0QTo0ZBnpIvY5kAg91FxAwWIxkxTQ3s0xQJF/PKl/wnmkYfyjkJ97K5Zn62wOPqPdieLqfl04SxBcbrOjrEU0cM6MQZ4YXJTVLgiNJ7OBUFIP+zqZkvi8V86KdbvpAeB+ssKdymC59Szj3veZ/7lLSSNwKrfkxRhNFBYNTgnyIg14YVePNrhxpUexs24h/HL4mzHJZT6dx33ZmiZmcqiRlq30Psp8kCSaKESxBUP4BnHLBH3RurXBqYCOFjcnwEbjjG4ozUk5qR7S6/zrF5JwuwL7+9fOTPCOJcU9miZvEuhJ5IfxWJcojkDJB4j3i5QP6pmyZh1HGBOCAQtDKU029ib1spMgn0AUSQRtp8gaVlsiH0Huv0ZmOVzvzr7saoKh7JofXmy35O9Vb0JHjWOvHlTdGZkUcWCRVhprJuu67KoHB4Hx4Cz5xLAAR1vZ2tCnfRw5FSHzhbqHcfUSdLJNK9sZ5Ol2vkzFGUP/SNPgbExnGBwihb0MFU4u0FcLLOTbRERs5uAbFAb8a8g8xOsygbXxurAjDseHVIYY3iAjMLxOfjUrgNIRQBlkLdw+RE0vJaO49BrG9SF744f2ivEPBDV69h+R1cx3ga+usfASCj/e9cNo0W3pfkUA51VwuCqZ5zPED59HSIvGQ5ngYAqtDnsOt/3rnI0ndVBgoa25fx+oE8HBVjhyHWO8b2cIzIxpYmBWcQapqQnTuTzEoDiVz+rEOrADStyJpMuf4E4frrugREy5ju0EJ5rWyRWjfkpsnC07Z4e7DqzwlIQez5ghpSR0moH36u1S4sm8HKKZHPKO/cDnQqCGux4d3IswNGcSci1WVAEX4HD+qHu66cMtrKa18mmuo1dwieN12b7CuhFXlRqjRcE4ZJT+lvwCAkz9uYMl6w2QVUs1ddOtVtkXdQA8BVgcibFabwwwhKK8aG40VUhJ4b4im2I4yRDmQdLNIBbwG6JJkU45ALHiBz8AoYMF87ZHkqIhWS14YQz8VwEoNYFEDZCScHD+nkN+RZyfmJB+IlEpfdjZ/nNtbB6Z5BkS8/Byj3l2ImgPj0+0QU75MtnOiJpmbtliCGmo1QAvzOCuksvDg7+1XcDnQ0UZ1bTjx2znftyz91iYCtqi6Wngj7h6oC5qrbJiGYgQOLito6lEoFuxxLz32Oudi5ONYdcZPaCis+P43u4z5fNbjWHfT1lPNeddgZgqZ6tWptXZetNqKey7ZtgvqcuyFf3os5wfH4Zj/pmeWmKpoSiSHuyUCyJv4JwqRGG3WbHn6UiVsPC5eIQVGGVU2pBmx/cQN4AYq3Y13XNqoMyzZF/43hzDxdQtm6KCDAfE9YneX6iDAgFpIzwxN98TEEUVgfc5hMpsREbLYUAvebVFFKsyaRncWUEpDvZizlqorNxRl3vf8EE2cJK3UThM0mxLhqFar33HCeBoL6UBTvIdkPoIONqj1D52MUCLKFXxpFMkdJDaBYiuLkQR0N7pam1RwdhN15vx5YB2N8br6zBtgXo3LwoPK8OKlhgBEWcO7nUGV5qev/gA95ruOMn7mC4HY+f9eUZVM1zlRN6XI9s7ArDkiZYbZGUFHYQlXBkInIBVPKQbP7woJS7SiVPM6dGD05EqNRs8WFe3LILimFVYhAw8x0rS78R1QOPM0IhfELesScxLUVb5DIebO9capaoNAAPN2GWFg2MHiz1JDk1utpLPGBi3a1/hYar2LkJ/05nQCRq3VE/hue724+ZXHThNo45AF7OaqQ/ZhL6cHyq7lUUkDPa8wpxhTYTeujliwAW/TMtNDMemF1SFRlMWDPQz54MJAp5qTijdIwOmsrY7HYCrtHBheBKaRk4U/jbFImBAnfakEuaYAUxDbMZFCQVSk2N+BVJ7Lyr4QjkE/eWMdGcOAHnVOSDabVl/4mwNOBaPZj5frW5L5idb+UQKpknwG1rbYL0O/K9+9FOZFVDxB4Q/PZfIELx0Y5AzJuHnKAnvEuBlYgzOkHjdpFmOb0QB5bYUfs8kN/OzJYM3OZBN2aIHxZCD4IKJGUPvD15hDcZCD2FdIlBUEdRGQbk92mpzg4mFB8aLVrMZr71pdmDXYhXLUErNtimCErxtX4BNYGx2EIfosQ2EeUCaRd37XMJox0i7NpM2C1bQ3Wyen0QEiFSUE+uBG51msbyyZhU3ku225pAifTI7c06itJ0JmqgIQ1lP3T20qmz6x8g9ggsLPT8qmT2TjyCCkvnN9ElpuVrBJhx7gaIKl3QJVLEC32KiFwL41Agi8Z0966ivTT8xCajwrrapd1Mb2M8WiICN3gsEflz7As64xJcw7kPtlo7NYT0DohYJr1pMcTpdVoTyLpsantJ7JOQOrWgBThNAFnAfEOmxxz16wiEzQoOQhxIZQTSb0t3PuwwK/xSk4CpPsOBv3tgv4pCD1ogECxFirXulBFEuBR25XmPgtN2vtvCS86G7wKF6GwgS0WyFR+GiniHomu/j/JFSceoV0W3twWD7qvSA7EmvRpijODD/7V5KAMjpSdqqHFvRgWeiNCjSiEFJx3pZQsklq5O6prqHHvBuvh80UM7Bg/P+iwbLkvHrZPReH73+ymvNILlDSB4AEZH2qkPuVU4BpewaKHphXywe4mhVFVKiYkDhz2hlHk+eJE/+8teDofpSaAc7j6Gz5Awdythbg0B3BtWL8b0lAJRLwHo3FGJAEoiD8uVaE9bxntzi2YJBlkXy4usvcMYgYdqiZLCOEJa6dkGbWhTwly0WXFZE1P/2/8++3fZ1uO/Mm3cQu1V5FH75bSqGk3uTLO4/xI/tdFoTcR2zyOdi5rHcuoancxI6XK82BwzOloZ3oMbGAux1SOYp0mm3eASimxLedd//EIVQdWBLWQLnm7Qtv8CZpz7U5atHSIQP5xeE0K5OQOr2hFp048qQ48pRbJyycbkGSjOPmQxdGfdtDsZudbNNnLKB+wMfAMtvB3LNBQfjGIQOXTsKZVIf9u9+qnPJpq91K0T/cmQNrM6/93RvWyKBAqw2jS40lb66JDiwYcOJ6zwUdTx4RrBHKXv43J1iCtu2sko9i0yF4iG5obEAkjmfJJPJREEae1T/nFJrqH92IJBPX4UuEGTiun5/++RAWYYjnRSVE8br5IqBmdg/Y4N0vJcLHC78acYzEN5yenXRX6yqOYK1xAcd9t4s1WXG/uuo1sums4vz72NTKH7Cbm2Sk8KAdTQfG7Qe4J8Gk7vd1OASpOmlTrfbwWwO83Wa7WMy27HDFJyVPn4Ab95hGVvVM9MmtMaclOjxibSkM5ev1nL7olXkhuYOLaadGzvEQeWXk8nwe/v7NdQDQkcsTb4GEMbuG0Km0Ria4ecGZj2dr61+OT7afoK/4iinIVXCVsZjzqT8ZGoWYEHvLUPiUoLNiyFpRKi0W2V+pZDpuEcEc5H8NBK8q3n3ybgBpngJRg4/fICscY9seKhqXrqx/eevnU1TOTyb2K0P0XFiOAqXvqVdPoyoHNWjeK8cAWJ8fZINpjKzzYO2rDrJCq1xuKTWIL7GMKSEW7hFO/U5NMVTUwAUTFopeDtI8KBAlGUm2Z4PcHDmMBkXf0NP4KDlMAinrLQqLi3ZS+zkQyrfMXt/NV3jjaT/btR/BRXfHbB+bso0lgXklZgiQ+oF/WQOFxXlbSB/IymCZ3jcXcJqQI4bWDfUdATRZjaAznwXk/9JrGRbZOwXR9LxIZV+SuMkdKgeY25BXkbAvBcIGDmvOa20ON1JNO1xHUZH3erY+EY/3mJpF90//uwMJZO6ne9jes/WesNV7G/6Ga9DfpxmYxY+tr20q4PTCxuKkDvoqC+1deIaNnCKbvU4HB3RFs2hqRppUDOlvNhSUUEHPv3XlvFCFd1bCHpGw5eHr52h9RmSifuvJrts00UPQR/k0fwNIX4jqMX9FglAaLzXbL3SWlCKScVCOnoHvyHTRIcGelVFOyaocOCoLtr1w1v4sTcOQXRspILjEn4ZrQlkZ0waGvy1N8OJcwcGXv9xiI/t6uPIvBxGHlAV0OiihsmEVSrGzqP/Af1WfghpTwAA\"], \"cfpb_seed_common.py\": [\"f06b311934401cfe065e54c58ada52f4896b50124646d179549a81f4d6a172d5\", \"H4sIAAAAAAAC/81abY/bxhH+fr9iy0/kRcecgzYoBCuAETeAPzQNmjRfBIG3Eld3zPEtfDlbudx/7zP7vhQlK3YK1DBskbs7OzM788zMDqMo+ll0fdHUN7UYh46XLBeD6KqiLvqh2LGeV21Z1PeM1znLR/ze8UGwcSjKYihEn15d/fQgesH2Y70bQKdnu6YeeFGzSuweeF3setbU5SFl/xal4JjZNiByYJ2oMKtnmMnZk2JC5Fe52BX0E+O7psvlvtuxKHPRpeyfYzkUN7umHKua9WB34Ix3gtUCBNi+5MMgQAQ0h4Zx0CqLqoA4N780Bb1/FIeFpEjzD4yXZQNxaLcdr9lWMPGhbboBM/dNx/iYF0N6FUXR1dW+ayqWZftxGDuRZayoaCJo1c0gKfRXV/rdA+8fymJrHn/pm1otb/lAA2btD3hUA8OhJRXr929qMPkOXPNtKSzVeqxacNyzujWvWkiCF/jb5ldXkHbP+gf+1d++zvZFKWLabil3WbDdw1g/Zn3xm1iSctiKvWKvX7OvbhN28w1pcnnF8Ccv7kVPo1qGVNGLEzn6vhgeJD1JOkmbVtRx1G2jhHgAEcErRUfOfgATbAsNP7LlSg+n+CePHTeJm+/2T8c2h5HFcq3auhPQe23GH8QH9QuMKbkDo81wzHEvRC5lXbCaV6Jv+Q6yg4sFu37i5Sj6JWu2v4jdEKqg5Yey4Tl0QAeX5lB7H1sm1yAoSScLWhJb0ni+XtMbSTuR9iN/kn2r/TabhaUj6p7siPe7olh9x8teuLFetByW3XT9Ko4W0YJFyyhRw0kq6l2Tizgah/3N36NAOZMz04IkM9rK6qareAn955n0orHKFI/xvoNESxhU+hbO9R09LbSnLVkJ9ZKQG6kyf47SXVX0PRkyTlt6ETQFbcnFWMHoSdJPlQf3ieK/2JuVzho6XgAqfiam/tF1TRfvox8NFHmOb3aUbgS8aEcc67N++6L146QFZ5KBtSKxASPtQVs3nZhGlsKAi+PH0VirSRvQOn6Z8h7OjPPBenAQJSlcsax5HL3+/s034Xm51fpY2q4hvwaY8DJz2KSsb+5glHTKyuXvo4NaXJ04KoDaG7WDYHd39d0d65r3PdseWMk7CQIKnwG7yu+BjwBHIHcxMAD/zRae/JgSNuojrNlrdnvmANX5CUZuT3AL4CTErcU9pHwSkbUFGpiq/5jam4FROBkQXATTRmzPrwf3v45FJ/I5E/iY+RuDtxwpoxVVOxwcS9qGQW99rebDU/kTL0rCbXLbIhe8pB/6LMENHkBsp8/YqljOrnflSJEvgx1s+ZYC7IEGctEX93X2XhT3D0O0cTpRZuSfbayZWhkHk5Pvu2ZspeRO8FS+3B5iwzp57OqnboT0Oeyw5gqWkpTOK04A3IR1sSchvYNLZyTBB+1EAxkPBRBgcUxYWQMYS4QJqcIk4GetFUSeZF858hv2pbeSXWvaIQWnWemPbbovG1jHdIfE+CVYmvAwexwBS4bLm9ltQ2oZXIPCT0DBgA1v2/LgYknJq23Oye+WfqzBs+EWC5N0aMibY+hxJmjMB4vEBRP+oehXr3T00Ohj/HpljuuGECSeky7txyrWx4ZsTC7S81KyGOM3LkCetG+nGo+7fifqHEC5Wmt5yAK9CY9YvooqQZCE7SITBqXNWdiWT4TaksX10u67ce5quObDWk4P3HLDvkBGNDHOWX8MjdUj8OWsDYckJ55MWVh66688sSccrS2RZMS3C7LxmtdBIDGikd9aBHD6ToKTMv4+1WzozURKokEyF5skjWJfEH5KUM8oRGR8t2vGesBxfl7IUoBf502VuSxOvr3W64OEDmqMQvYkT9pWijxTKrFzv20wzCkLfvc20gFyQGUj1mHW4z9pQ0IssPRknILNBRnNZQnMu7depDKpD3u2pE3i4tUnq5PZgQ5Z9bmoxaSTDLFbNuflbLUKIriyLkXmgfL220QnTAuPtc+Nr++b7lEljUaTF6ZkeuF6H2V2m2ejwPn0zN9wTctQkkqwsphtBu1ZbNKKt0egLcVZzpQdnt16hcdCLUg8GAZsD702+8DSKKHwcCPz8oxZSWe1tJnYT7Y9EHsg8mxFkTZPeEAhx2gomSWXLClM2ZU0Bc+ydLdbIFBlebEb4qYrRD2sIlXA99qWX+yJyiJcykYEtMJtPhIIfTotcWYAcpkUBcJJafAiWSTkAZQ+mPJjRSWvwFJP7rlajVY7F4CQq2M1ru2eLhsb+CNxQF5G6vTdy9EDT2pe4GgqmyTgHIVLLx+aXtQm2gQQfmS4c2guHZY2S3yPcpTXkbnayU5HOmRTXMtzYtIZssfRziM3GXRkpF9QqoTEIFY0TbDrcftDQJhDohq6jeXcBQOhBnmRjFw6aEll13BOgCV5jjbo0LY1RRyKepf2AxGkWEY6jpJNuLH6EYZZs4lFXbI5NTFhf1F5cMU/2Fz4NsyHz1U4P3iA74eCvMhl7CH/qnd0yYLLLmUQgF8q48I6U3ETQPYlMf1/EcNPFKOG+QXLpoHuwlTDsuoFwoXPQJAAS7WYTWdvkDQKGeQ/qQUfrdTtknyNhZ565nOZya2Un998es5CALyWl1yYu3FZCxhCef7qnLm9tZe7NLkae1met01feOX50Q3Ps6+BheP85fPveyxDN8SQCYPerc/xTc+ZXOL/I+o7Bk3c+6PAbkyYywttRwe+vT4XS6dl/W6spAuhtC+HGBpOgtswbwfzc1pfTHidYI5as/DyjWgrgL8iWnoA6Cq9iO+hVT1olvvjKOqaJ4RTfzmM7NR0KXsfSYdTs9e+OjZpPdbFr6OIA7X4FKATvZy0s9BZjL5nl6KDFvoI/F5k/YjaXlTIe2INAChr6RK7n0ONAOGQJM1Cy6cURh8FDunDuBOokEFSpFDdgN/Z93SLtpL//bGKCBeAP0pdyHu4sHdE2RNFXMF3D0i9ivsCNTHbiRLhbIu4Pqhr2+FB0NWtsJeJVnlkV/35gsIpOqwqiOAFy49OYkLFciLV0avsQ3ah3lG2kZIA2V5dvQVce1zYpYSHp5cbht2Nj6KBvhpZmXRFJ630d8pDTRnqb6WzbPl6IsEm8HJPMf42ZxRmp4WKOl+p2UWX1GqOk6Bqc29nEdwN/ykYHs14djSD559dotm8DcXLsb4+LTrQHwnWmW3Y9tO66lGIdhXti64PVnm24WK1hxd0YUEGJ/GCGqmyxNIyJOwbCZf+gnMZblAA7QFVSuXMqRwthx3OA1G/hacC/Q0zCP3Bzi/Y+tnf9yWy1JNA0RfVPeqS7dS648LGm+8yU+nxenGg/cAF3fzUmZGykMm4d7GYOSCQ+8+ZzomCOgh3nrM4G8DedQ+LrdDFgkP65kGgY2RSd7D+uiaTTerAfnz2zUp183k03HYClo4Gp8hMsJAzJauXVXTWmz5e0wU1wIma7sSV6MI/tJO3pruS9z3L/lPDwL7DuO6wU/sV04shy8h89zLYkjstvQZwuU9R28IBll5GTzkAXeAgBzGU9qArqSwMrlHLIOhnTwim4BYrOWKQuv4IQE17vDd/LUc3VMqqHQKnnZm4Ui8tZ+FwMr1dPJ7ipBtJdVq8UuwH3b7vyO2cpKHuaB4AoBn0PPk7YIpmUPfevpDzAvktERLbkTktO1X4szsnG90Lmx/V2eR+/O23AxwRn3XUIJcVeVyjy4/szHwG4rruMx8qRL9HKX3gEpuuO32YgPXqZot+kW9IisnZLwgixcjT7V+ziH1xyQcF6+WrrzdaCvm5DOK3InJpBd1yxKC5AcJmSI8m0lMh3s/N2HZFjgh9esL15dW4iR1jbUKNdx72u5k/rzzHRz4bl0K/oc4g+/YGAUgoayCmEfaoJjKakPHWINaNEp4p4VPlNG9s29uuqXj3iEe0+Bv0+GEOd3cIEaKjggkvCmr5twAqzIGkWIovtSwtIo0BvZUmxevD8EDhrUHi3iF7x5cBd3cH0StyukxjqtjTWbBWpKreO0Hfcw0p+5f86ot6Bt4MLYH72MBIcqCPzbS+NNka+UslcydpRTI8eReJeLWOKl6PqieEFFjQMQ0IK0GHFf9g6XvRoRbGt2d1vI6wGfUsB2A5/f/K3FAGoRsX3feFCkcO5vWYgMbM9b6yBIvh3g0/+SchgwYFlcdrqdbraGfMCoiQ8Uimpd6bbYRsnj5vU6l9rC5AdSOVbiFWhI1eBkZMUddXXYArsJBoYTCRegESDZMkCVbZG1l6cENGA6mC6ms3KtGGrsslcPpRTKhqVocyowWDUpaijn/eBzgeQT+K2RUSzCXuyV1QP6U8128sTxmyFLPCeXjQF6Fpy5OYbFhYz+y7Sc7J4rVBDLYhmUNLItjd9GMif2uY39KNaJM2RBA25Qxjw3NtmhBML2rVKFzIPNf02XSqUxmYfkKgJpH8LZWMZKdVH6u2jJnyHNUNWbVFpOjFZ0DB+4yOjg3Yasxzl42x6KMxOA5Z+LFKJyhxVqVB9LlIo2K/p+rtScwBhsbNzKBuJt0OM2+90zh6La9XyKvBkIKcmRNanbH8Na1a3242fv7jlpKcUzsI06AJY/Y7ipONLdA3HIcnLL/09d9I3mTqKfE43PeEvj6+vz2FY9DSkSnTydfFoOVITlEroOijllvyUdgyO/WqZ+UT1DcDHsTrDpRK9tjrVVCQn0xuko80I4/tRjrkGbOioc1x/vniM4pMIp6gBL5Gii2RxB59CDI+tnocTYQ47nC4fEOnWJYgPq557F3KodBAXzrg26/Lmws2eussWIv3CU2GiYG4drWEUK97PWN/qs+9j6g1UoqhqZfPcrK+FAlv+BeUambBtcXZXpgnr39ZfkJwNwUEFt5dkLts81oAQQVB9+9u0GptZX8tjpt6c32H6+tQwFmui7odB4UhusMgf4cNCB9pMtePmIcib6XFSOd1ZqcQPs+uMYwdAXLIpMKHo1UBcPgr9Pe+2dFKLKS6Nvbw5AIoAoppjFu9sk2T/wLipTiqZzIAAA==\"], \"cfpb_seed_v052.py\": [\"d08ba4c380a73b99d702c77ab82e8b7acd527fa8006c5cce359cdb107daf85a3\", \"H4sIAAAAAAAC/9U9a4/bRpLf9Su4BHaXxGm0Y1+cy+lOweX8WBi4TXJ+LO4gCAQltWYYU6TMh+3JZPa3X1U/qx+kKDsJcAZsS2R3dXV1vbu6Fcfxi7wor3Zl3bJ99PTFj/8ZvWbw6cP1k8XjaJdX+2Kfdyza9kW5Z81iNntzy+Rb+Sw6NaxlzQfWRnlZRixvygKelkXF8hu2iF52UcPe90WDDXY7duoA/p7tiraoq1nDdnWD4301j/J+X+DLQ//zz3fRTZ83+xaeVnv4G/XV7javbuD1qalPddNB77yMoDeLTmUPsGctO+UNYFveRWW+ZWUJjXf1B9YAHlHbn04lO7KqW8ziOJ7NDk19jLLs0Hd9w7IsKo4IFYaq6i5H6O1sJp/d5u1tWWzV159aQJx3B9rkuzJvW5ibfKkf6RasK46MvObf5xH++3NdMdHulHc4hGr2I3wVL7q7U1HdqOffVXcaqxNQJod5t9FpL6ez2B1O26yFFcxgjVSnZBbBnx9fvsx+/O7Nm+evvn89509e//D21dPn2Zvn//Mme/rDf739m3qRNWyf7zr474Z9yk5FIR7nQMG7jJXFTbEtyqK7y5q+ZO18lnqj7+rjsa5sBOTyZnx5s5um7k/ZLj8J2HvWseZYVEXbFTv3JV3xrO1gjYtDAZDa/AhrOqVN9rHobjNgv7qvOqCn6NOyksE0FY9khkfk+9v88ZOvs0OBg6Sz2ezV8/9++/LV82fZs+dPX75++cP32Xevnn/3OlpF97xDzD4h3fb9qSx2sNJtLADFHQMs4EF2yI9AOvNCEMPvAKQHUrBGfQcM77L2FmaYlXV1Y5rhQsGk1QNYLPWxBHHpYVr6O8y7F/Q9sBy5Xo/GiQSvs/c9ML9Bov6YIW2aYk8xK1neArF2t+yYw9MHIMx/GLbn/3It8vfrJ4+f1tWhuFnyroArcEyV7YtmKThcrLxQBZlQBRlKAnld992p75w+DbB+feSstoyKqoMFeHz9+Ovrf3n0zzMJ9ACyfarbLgOe6rIsgbU+pNHVt9H3IHMCH8kChwVBDCDhKIn7PF0AueryA0tSu28IfQtIqMEgNDNbC4Z5THvOZnyaDctB4NoPiaEcn+hpv3gGq/KiyY9ywg2DVa/whe6jh8fOc/1tD2qHrWKQo0IzG/55x9gpg1HzvuyyKl+9yMuWmdes2tV76LGK++5w9c1VW6jOBFuOfnaDjFXl1Y7BCsHkEpsZzFzmEnUk3jLaF7tuDWjNURVuxDuUzwonGeGLGZ889hSTPrIuR+4EggogixvWJbE7PvB3dP+Q8pcKoFic4hCBSdBwDO80edGy6O952bPnTVM3ySF+JvGXQ0X7GiwD9kYTcrUtwJaZgSM+8DK6V+M9xGJEyUMWMRZg20AxRX/R06XI8RbsE0h3m6Quhi+gw/d19wJ0314gynlQKOVd14MZXVFdR14DeNniDytNAUFA0SFOR+ih3/BViv/qzJwTJToW7THvdreUDv8WxU5f9ukEqprtV/cWFn8WWPw5fZhLRFf34v8HAyG1mB91iODFsgYhUC4JGMyvRniQM1UHapqtHRYkLGm/2WyWhHWBxOg3LHDQNuGibQ2UCpns2KcuccQoTvViUA6W+gRoikDiFJcohlnEIysSE+9Oe2WHAmxm8TNxyyLjlkmWvEEv0JYg7v9l+EKIjsIRn0gW2dUnJhETrZVr8jjTvmUc1Q3nYdPvhB7VfpS16EQ4aIFh0UpxEABCZGvrvgGhR93uku7J4tHFxHv+7Dv+6BERbC3GDXrVNtW4y2QTrIXnoi1HZMDJGENMqx1YsAhHiEBZoG8IzjHIV4seJFC5r5QYSQw/wLrjGogl4agpTKjMCPznckJSekJ9+b/LYYlwDLDwNpBA2G/tuSEb4YV1QOc2043FB7mY9J0ie1PUDXin4abqbWwpd6Be1Xa4dokFUswgVSxqvRtbkFdA/bq6Mn6o7KqmjAsjWYiwaQn62x6fMwR/rGcFqKw346IhhtI9jn3bRdwzhciI61oAApGbgxUY97vWINP2xwRcq+QDQk6jA4zLP4Lutumw4I/B7nBksYvGTTRYK9qDf9tlLWgaubJTZiEJxucAKEVdHXnQUumtg3d8wBjUXnhWNcXuFr36TDcB1xAc7jtHDtVb0e9Y75XygmfgCheHjH26zQETUGL5AbxzKxgChYYhSzuqQt5WGGTAOEB8gxjBXSI2hNQBwvUMTAYX5TKHIHtvZgXYSgX4punZOBq6exANSXAcLRLJAYmRjqdtGgdCKJu2qoGWRdAcN3eSvKAQshNQE2J8bAZTEnSFuI237I8Xq+XuYw3Sh4iaDIDULoa2yKs2YphO4DyV6XwDzOPasLbiaBCuY44hGRo0wdKj3PzUz0NgegXXCubbg0vUQWLFS23EAwSEnAtIHUPvVZHKLD53yj8TmS0ioWBHBrbQrEVhlDVGmhuFHXxRTglfgGwHQWLDvWtQHDKR0BTtu+ks+hxD3oj3vMKeEUSjgivlGJEew1BJ47HDuLM5wuLAo0zGyILb2Kdd2YNoj1NIdse0iRgVKCN77gPjAbsgNqgYLh3rreo6NpYOw80C2NG5Xgr9yFqQ/XTCvwI6m+EUPo61EtE/QcbOCmhsxHfLAhf77HKkpDUAkQZ2jF4+GzKhdLxjXhUHNBLoX7dyRQBRMMkQgHZ1phqMLs/fZCMeORh6oJzmQjRkZCtwU2F+sQeTzQ7FJxGVgvweucKFhASEsfX2J4Z+BbhD8FYMvy9ucJiVSjcuRGyDDkFCe6cLHh+wREcHi1v2SfRO0vXy0dcbK+DhWET/JOFrbHnuC6MPMPuJzqTMIwzA2lO+Y5MRP+V3GNUA5of4HiE9/HKvocBnCuEhdrGnuDozl4Ct+Un8RS4SZVtrgUQaqObUgw9Ksx8iTaAbui8HsyUtZDlgVgLkAoKZuyQlr9agJuTMgCPjDff0Qy8WeYv5lETlU1IbHxxCfaaj6IeBgQbfhcdCAzLYQ2cf90m6yKu7ZNSSPVOtIwqmRbcQ9ZerlFF1w7qj3sI4xwDmlIJweLdWRMPVJJZis4B8QFnlCbdn6TwamClMNJ2ZON9YHjPqWWohDBMtKpT/sLKhTU11xBYhbP97x8ADbCVPCfPGB2z9nIeM3Fb3LfcXE4XXlYUUpj78bAm4Tbqf1fxKQUm9DInAKUMhFHZGLBEQEFJke/YpsYmXrmNwGuoKmKHkeYtMpoT8lcHgAc038IjhdFDAGFwlEKSAA7KS+YyWhv1Uc+BSgiYEMO4imhwq95PIJBZ5t6YwNrikGsoA9mT8gSyfvbwioeAls2ytl9oqRZL2yJobZnjHsKmFgf2NM5hFgLn/3iw5WKWub0NtlMm2NhCsFnzHBq0k5NO4ruVbfejaT2iLHOCOu9HfNpDuQhthC86uLvtj1a7uJ8xoyTc7ss+YqdczSIEHIx3mRV2thmh/W39cxSU7UOqoHMmKhzjA8PCfTohTQxKcCTUo4QZKQ+oBY/ClqA49O5Sc+vBQqsHQUNIBm3mJVgHOMteWx3rOXOuGX26uqfNs3ONQg4ClHXwXtrRSaSvjIzQw2qBBODAfbqTCvoMxSwTy6B7ED3Igxz6jv5xvUXwjvlurKH9P4D7Ev6MlMCvx/9cS0FjpEkPAN7h5bpiEa/hse+fQds55aIXxkVqb6gAOWIfrsg7SRz9EYtMXczEuEl8iQOnFH61jLfBVXxXve9hoRFI9mhn9zWkrkRjlxKeyFaafGxpVcqpICLZtXMeqUYY6AzKnXAi5Cxhu1vBUK28Vx7Np0zZYi9lKjnIpUKB3ek246phDHmLQy2fv/VgtDTm8CAacNp47IgpYfCQdcGK4zPEv8eKnuqgSok4krqJRhhGV481OHZoS8GAlFZb3EgXiKfqgguuF7DqlDxlbjjVoQjKZPkAQZ43HuY0IaUXMUwDxmoH6aeWmHc/4Ao1UEYWKJGS6Q9dWbNaxaiqMMqZiFEPL2oysuwXMb+uSh8gQ0WowpnwD4BxzqLWpT33Jq48ykws0ADlVjHPkRkzLyHWnkD7nAiury5+sTgE3AJhcpsNFBlzC1RNJQ26TmWiWb2tMviiKAM5ywJnjsqkOIEs7hk1hMEIT4kneoO30aE2HV8sl1yhD10UuGyDwDzVlcNaMa4VbQonHBhQqr8nh1Th6EvF7cH9xM0Bvb2amlRvQOpB48v08KNFsGFZf9S364KAFwQIN8oQP/QjVMuCkQUqYNTtwl0f55BcNaHsHfQEUZJTB8ccdwgy3DVARXN6/qLjLnLEWdlKmYtCwE8PsRXaCLCzs/dsOq9NVkupBahVZAibVKK0b49rmy/VMqFzJKiewgPK9vnW4jmDabilXj9DIVpdiBnPRNSXFb0i1vAE7LkamyKBSXm/stuh7oh5E7DZSJTrghL91rpWqgxtoZyr1lkGSGNS2wKnG6Ucbo8XZsq9ccal0fLqh3j/4M6GNYqOK66q8o86oHBP/4yPiB1CYpoPeN9lYA3H2yh2/DSNex2PDR+itkO1TZMu7bHsXb2wXVzXluMjwWbpq2heqDweg7FxaWGzNqh5SDyAzibNDq3fHrd0sCPIYbnpG9h6Yvfm+WQv4vrvEWVC9hbhfhBAcY/4xHRBVrZpaSmyEZlGV88sHUDk8/b5lB9zDg50T8Phg75xViYEzANt84SNYauYf5uVY8lCYCyonrl21AE0IlxyIQqxGgXKZAgNR3FS8vHMUnhLAlBhSh6gUtlInsFkg8s/DVDnmJ9vklPlxu89F9cDS3XkgulFx6Fw0TZ30pLdsjmEzi4huciarEizMbfIt9uBOkNrbZGBZzvXy6B7okKjU1gA+IVYWDtY5TsYSvWCXfzeCeyaO9TJmB6v6AuqG9Ahcoci44GHpJJ5FVz3q6l5/xPI8BQIq9EL4PtiwrMJYzrJAg/Ei7yF+sPN6Gif7sdDNTlPDm7B/9vg6u76+hg9SnwbWDmt4mBEyHjlPXQSP5uDwol5En0ERPPapotWwF0qFmjoRp+rLDdx4eztMpKp/qbHzIAjfYgGRIav2hix+O9CXi/6Eq5WY0c9t1QTcDh/IBE0bgKa0ow9vXMkSMdYejJq+tfz3oSw1J/BSacFQC8lKvFGQjUUu3rWGu6bWHCKkDSCEbeYoOCGuZ6ARmQ4AU5SELpaohJrquiYyXUiWjvR7cIRS1ret0HWHjBMo7MTmzXkEC4mT554ISbMpUeYAvPIfv6BtSi0baA1TES4K8QZq2WRZonh7xu/IRJ5uwvh+QhgHEd3tjXgx8JxwscrCHDA0yPjTNuFf7LjIyavAYJtlOHuBuTIwgikPiRLH5KOZgZdzU3PIxyJRD/8OxOANFEJADzhq00onRgV5EEvKY0naJiQKJ14D92uGduNnWkZCPwe+oNrHunmH6dOVxpV6aWZ58AhSrKSdFyE6SyUBpUJCxDklaChdehlSeseXVFoKSx35O1SzKh8lQc4ltH2Rgyjh+aKWH14IHugyWlD3HlDRc4PkPLK8RGGW9Fu692QIAnk5UaGpm6HlCtNEREpCsXs08Y9uEaKITh5BJKzPpohnZOYEvSA1zOtxcpB25+jBT6bxYnKdcvBOq21kAVDRZO6hFcL65OyKBgoI7X+CPeOdyNwAhJa3lSAhTsfjb5dDVR2hiLFgHynILWScIQV4MUTZzwdolluQxVrtgdOO/mrrB/Qgl6RnGnpLSRNsQCZK3sPgK2RrOjXFxqZaUgIHjp7PAm7wirIeifM/Fcf+CIYIXJ0G8rv8AOgJNrsrYdVWVlW4zV0CCzUpXnuLhzXhyFuGNcrFDsoKBNi8JNmPdD7O5xz+BDanXckK8m4k087VCChVT9WRzKtUBsuA+NNWiBk2cplGJSaJsdQc5tth/j3Lu4xXOSde1ixkt9AYw4El3kNW90kLDp+FYYImlrnGhSOuLCbIeHdyWE5YL6iRtlOqpqxcJlG17Q0W531hihVN/LTp/+4ZWVHRrwvm16MHEmQSF6wEr64VT0FE8EF/zLTzur1TDpCdWeRnRYJ5Q30m6Vyyd3rulSYXFR/I4yJWklFuWwbmrwil3LZ0JO3oeIgmJXFBAi+Yo3U3wEh7z9uELDL/OJI2+8x8hZ7PpHTFWqZ+7YSxZAMrM6zhqvTwZizJ8eh8kkMfGbXiMGMMBKvKRROcTBWFnhAJls1ZFnnARuzJq5FWCDQxlOAj63nZQbc4zEnPAjlkFjC/XSlE7bdwQDeEjNdICybfeEcULzwkFM4eSexI6o6PhmGaP7fLs3pvK30DR+jUDx7BbULnb6cm9ux1gXeKPfSp3HkQslwKONwrPgznAsXhTY6lveujN30GJdbIuXWFha6ysAlGRxLbSvRJaHNJBGhxOlBeYeeE7ajVfqfKoYLZ4YvTSg5BlsLkf0F2SdJBr7PK5BDyhLI5+kiXE6WGHBn3j23dgcAUhnRivI4hJNyY8HPHd+FcjgMJxL4MCQLoM7DQbvIXIqHhXIIDyQUK9XBBEnCwg3KVhKaTDCofjsHXNwFA821dl4mxPmZcsLa+Hh7mdqFiSpMuRY6/DxLWzoZxAqcmC8a/c6fKV11WystKhobwUizLsWKID4Rf4bW+ngcfu3wyRTjTIKSr87AGBC1A8tAqGNn4Dac7hOKXTdkX66lzVqL4m085gOIXztnVIuen/BCwkSObTuYGHLkbMLgJEIq54fgDZohEmlIeKhCEDSTCL4lei4rXpXDAQvucL3MXngmJPXiVn5Wus47oWmO4DpUMF1YaiD7RKa4kkQ2IF6NqYLmrg03oGQILqq7EiuPUqwc352hVVlCcFHCqCjeOrzs2OH3KixG868zCh3H4Nv3MPYoSDrQCV6fZUZeASUIthj45bP8LTgscAnEQUeMPnK+FZBWWIYCtwDtHSklqiKTn7kjDhzIyfYWHxdAhXo70/UrOFR4waUJvcZeG3KsZLQ1/A82vtlBdymvDxchQKLJ7h7tkEhyYSnorEgxFfW+5wIS3ADv8W5xgGwgd8vj8acZD/Px4gisyNETE3hoVMHvHY3mZxAit/J945ZoSSL4QKdlFRACjOLytlLCpMwQSisEGaCEAKbyQHyF4Az8A72XpkA3kEUx9y59yDJZU0gIkRLqBOsToBa9/4bBwxcXtXh4Rh45CsLbYo8MkzjTiGTmKvcBVYa+Py2tFZuVtJJIT65ahDIp0+Oya5Q3dedUILhgyyJnlU5OxbytoLWaSMnfDKkyKYWSI+wmJLIv83D3IASsx6Xg+ji+2xnVvo/9sL1XwS4G+MX5a6+9OMskMBDFnp1vbT90+kCWDGzBM6x/ld7dd228zt+3rfnt1GmgP55p6plu+5N9CMO12CLEIt3UjaUkKJ+Pg9CLZTl3hITr6b9y+cIcm7Dap9uKbT3FxUyS5UcuQ3Xvl9iYl76oXrYKfX1iVL2GEkgtT6/m5KM4D3FflTSOuboAyXtacOpsZiTJrITcXiF+v+RZAImRgHRNJlAAzuPKlaekxsoDHqfIfkqfxNOngvbnxaFd6krUpPuS7u+x9nqHXal+6yLtu4dqEW7hy5p1JXC8j5/pFIU39lu/OWjnucNNjDte5VtzzxivgMrxWDyyZ3/qBeCvqCjG12RAgp3ZG7JutUN9o+4gVNQrUWOnKXzVYoa+UZdwXbX7TMBbhZqG5uM7cWCc9Huwkda9zBV2yE3eTOneVOgUsWLQhvU9ny1hW0yMdvIsEBejw3Z/6LlTrfhKezeSd6G2kf4n0LSYLvD4wNtdCmesaPVicuM7Ya8V8uqUqhZt8sZxTuoQXFsFJ0p9ZxS/B05ep+BO07z700J108yHtJer0f9Xb+9BA05PAak3O3FeqDmQOrDW5UNXhHYNn8DIfcGVyo+vdiy1+Z9x4GLmO7fFt3NAbVbd3otezdmH4lSPDBSDzKZ2DdR6TeobqOURHIgoTaHuOvr6+sG5uNcSjx9NNFYfFif6NMbQehK6L5GN5v8hKX/YLiQtIYnZJWMnkoP7v2oKnKfeQghH7KrJPbFVEindc6B5fP8mefPOv4/GVMhfQeA6N9c0n4B/DwVzQJvcE6nJuH0EP3XwTfDFwxY5sKz0nQQ7cXpG3sFNBES15OAzrxYoPokQUr74jQfWuhsNxWI/Vd7uVPuWaLvYdeh4HDjT+4/9e/fFooYCbmeHb00NHs5w+9i1Hqr1+EuxjX7Wg+ugnkktUmaesQrXKCSedjDMjywnMlUiQbeWZW49LbuY5X+GaWpWa9qaft3zDu3I0WhxnIXJYRmMrI0TrdiZMMIJQ27lMXowwUG1K5+FoqrmrsYN1Vme1uVzWyfeqTqwZMUUTzrEppIDYfeUfp5yBkxcnztUHfg5XuKtnCybcW/zJ7eUhFA3hzt3cOJ8NHQMZWw59x9mKznvnpvek3pTjfvFNkj+K9jz0kWdFPuatuqLcuaZTJYTVYKF7OjdWj7npG1IHAQCTV0EtvX7AGc9C1eW/aQuh6g/RpqzopZ6mwFC/10Clm6pj8Dn5HJy6W9BGlICrCweF2973ODs5Kc/yCj95YclQQlhfPJiJ9lp8BdlJA2EJ1eOx43vu0b3wFYfWgT1nUJ0ookMSTFwUrUQRHlFyRcvuELoI1b4wRPGCO3/z/DchgAFPKRBExkPTo0FIbJ1e5wlh+NclBX3zmxCDDkDJMYBSAF2PJOZ17DU/Twvldth0GHBGfh0aKODaqbCGdGdvnnszF6/imc4cW8eb1trAGo1uaLMZPO+k7psdqKqaeJReQfE4TD//bfiLDOtzirpJ16JYeDNXHkMWnizdMSWU0mfKBkBIn3ocCCF3GIxqEAZEdkAGdvHI4ZB4vKHEV/GVfQANXXR25jAmOMuQjcMLiohlvMS5P/9DD1gE2YYOsSGPl1Aw4GBtbVGYM3L2Pf6WLE49SMcrjtWG5eAxOv6rDuAb6zu9Bk6WL9Dhg+1KrNBIo2/t2/MvPJR1npTPsXWkIaE7Esm7WgbQ9o+2T8F40pGp8+i+kWDkeQobXTcyUUuyEjNA5RXce+LKy/UpbCVmvMdLYIZstKMc6X7PBZCJqbMBmpQ7qhF3Z5HoMp0U4m6m+TWpxfEd/JuIY08tV0947A7c8qx+R7SVuPraOhDDpUsliJY+aMwqWb9G91ink+YOFMwNlxNhiLYEgrOUl6GUOb1DKAbW9cJBAhBCAxH2uAy+6RgksbzZ5SKQ5FhyEFVpnC5FVHYLgSTcOwJUt/eBEwA8UXtmkAuYzgftc6GXzMDU7FQm9DovsHOIB03weAH8QG8ywIP2ijAzqtLEXOLXtpSj88i9RuJ9qH5IEauTIBF0cS/1nPOfBlXKBjYOsFKj3RUFBesr9gByQ6IfwtPV6QFwY4IeAumr8wDQgFAHycjlLbQAtvQG8VBSFRjdldNQf8PdAQgBsTwPgzOD9cuCNqTP5Q0/v0BlBgZWv4JI0mqORNrIz4O/WOhwDJEae4ghqZw2Bv11S14sNGZexVEr67oL23aGGgybRp6P9EQsnWz1xI6NLVHpOVOGnXyZScdMFbmBY9z6cNjycXrOqmBj8yIdtw/DjQd1Psn2kpfpJH1uUda8TamuJtvq9zO38mpqPUp8YRHKpAKUeNeImxhhdwzeqm22RVV/TNTvHS/gHYaMtbigMrGuCTUJUGPbginfWPismVVMIKOq5ZnqCB+Ku6MC1Q5hYFM3f4OXoPo/Obr0qzT8RhsqIbKACBPZS7odG+BL0kjtPZwRcAuseD0g1aSleRoQZdJuRI4ptIAQW3uZ+pphdVBGJKj8W4gX+AN2qUUYuo/qHcSJTaWpQogOMF6L6g1mfrPAzM+7BMG9ozdcuho8UzFaz+ofmhD4jZa1bUFRvbORnXAVsE4Dhikg6soOPRyvUvoGVClqGP3LAUt+WfTkfvKnPf0StTCjGhWKlzeEnvtMa/XxnlEZ0nfn2hs7qLsGdnpi+GE6cQgm079H53GitQvKTfeoLH+mPA/b+UD6PFxA2OQfrRCawS8HwoId4Df8hIh98+grpwtYpEdiTuq3/vxeXz35Oryw4sdTl+62WizcKP9wnTi1EDgUy6t+luJnk6uB47CCeMRDW2PLTfAMYcBUOKbGNMYbTvDS41YhgBY4weMLfPt36HyTKrPi6IqfiC7Ezxn7JwAfgnZbSdKNqBF2mW5AsIxyQC9Fii3ySdB8DYy2CZQEh2EO1QE/hDwgp3LSjoJNABwoobQLED/CnbciLW60La9Y3PfHU5uoxujKt5AFpTGJ8PUhiHmcep4+vYpJxgy6SnLDL8YlSLg/Ut3O/g/trRc9noMAAA==\"]}")
RUNTIME = Path("/content/findisputeeval_v052") if IN_COLAB else ROOT / ".runtime/seed_v052"
PACKAGE = RUNTIME / "findisputeeval/curation"
PACKAGE.mkdir(parents=True, exist_ok=True)
(RUNTIME / "findisputeeval/__init__.py").write_text("", encoding="utf-8")
(PACKAGE / "__init__.py").write_text("", encoding="utf-8")
for filename, values in embedded.items():
    expected, blob = values
    raw = gzip.decompress(base64.b64decode(blob))
    if hashlib.sha256(raw).hexdigest() != expected: raise ValueError(filename)
    (PACKAGE / filename).write_bytes(raw)
sys.path.insert(0, str(RUNTIME))
from findisputeeval.curation.cfpb_seed_v052 import SeedV052Config, build_seed_v052


In [ ]:
paths = build_seed_v052(SeedV052Config(eda_run_dir=EDA, decision_record_path=DECISION, output_dir=OUTPUT, random_seed=20260713))
manifest = json.loads(paths["manifest"].read_text(encoding="utf-8"))
print(json.dumps({
    "release": manifest["release"],
    "primary_rows": manifest["primary_rows"],
    "coverage_supplement_rows": manifest["coverage_supplement_rows"],
    "enrichment_rows": manifest["enrichment_rows"],
    "stress_rows": manifest["stress_rows"],
    "benchmark_eligible": manifest["benchmark_eligible"],
    "manifest": str(paths["manifest"]),
}, indent=2))
